# PhillyStat360 — 06: Vector Tiling with Tippecanoe

Builds vector tilesets (`.pmtiles`) from the GeoJSON exports produced by
`05_output_analysis.ipynb`:

- `data_py/vacancy_predictions.geojson` (≈436K parcel polygons, ~360 MB)
- `data_py/vacancy_predictions_flagged.geojson` (top 1% flagged parcels, ~3 MB)

Output goes to `data_py/tiles/` as PMTiles — a single-file format that can be
served directly from object storage (S3, GCS) or static web hosts and consumed
by MapLibre / Mapbox GL JS without a tile server.

Designed for Colab (Ubuntu). Tippecanoe is installed via `apt-get`.

## 0. Setup

In [1]:
# Mount Google Drive (run once per Colab session)
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import shutil
import subprocess
from pathlib import Path

ROOT      = Path('/content/drive/MyDrive/PhillyStat_R/PhillyStat360')
PY_PATH   = ROOT / 'data_py'
TILE_PATH = PY_PATH / 'tiles'
TILE_PATH.mkdir(parents=True, exist_ok=True)

FULL_GEOJSON    = PY_PATH / 'vacancy_predictions.geojson'
FLAGGED_GEOJSON = PY_PATH / 'vacancy_predictions_flagged.geojson'

for p in (FULL_GEOJSON, FLAGGED_GEOJSON):
    if not p.exists():
        print(f'[!] missing: {p}  — run 05_output_analysis.ipynb §6 first')
    else:
        print(f'[ok] {p.name}: {p.stat().st_size / 1e6:.1f} MB')

[ok] vacancy_predictions.geojson: 405.1 MB
[ok] vacancy_predictions_flagged.geojson: 3.9 MB


## 1. Install tippecanoe

Colab’s Ubuntu repos ship a recent enough tippecanoe (≥1.32) that supports
PMTiles output directly. If `apt` doesn’t have it, fall back to building
from source (Felt fork on GitHub).

In [3]:
def have_tippecanoe():
    return shutil.which('tippecanoe') is not None

if have_tippecanoe():
    print('tippecanoe already installed')
    !tippecanoe --version
else:
    print('Installing tippecanoe via apt…')
    !apt-get update -qq
    !apt-get install -y -qq tippecanoe

if not have_tippecanoe():
    print('apt install failed — building from source (Felt fork)…')
    !apt-get install -y -qq build-essential libsqlite3-dev zlib1g-dev git
    !git clone -q https://github.com/felt/tippecanoe.git /tmp/tippecanoe
    !cd /tmp/tippecanoe && make -j$(nproc) > /tmp/tippecanoe_build.log 2>&1 && make install

assert have_tippecanoe(), 'tippecanoe install failed — see /tmp/tippecanoe_build.log'
!tippecanoe --version

Installing tippecanoe via apt…
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
E: Unable to locate package tippecanoe
apt install failed — building from source (Felt fork)…
mkdir -p /usr/local/bin
mkdir -p /usr/local/share/man/man1/
cp tippecanoe /usr/local/bin/tippecanoe
cp tippecanoe-enumerate /usr/local/bin/tippecanoe-enumerate
cp tippecanoe-decode /usr/local/bin/tippecanoe-decode
cp tippecanoe-json-tool /usr/local/bin/tippecanoe-json-tool
cp tippecanoe-overzoom /usr/local/bin/tippecanoe-overzoom
cp tile-join /usr/local/bin/tile-join
cp man/tippecanoe.1 /usr/local/share/man/man1//tippecanoe.1
tippecanoe v2.80.0


## 2. Tile the flagged-only set (small, fast)

Flagged parcels are the top 1% by ensemble score — small enough to keep at
high zoom without dropping features. Renders cleanly down to z11 for citywide
overview.

Flag breakdown:
- `-Z10 -z16` — zoom range 10–16 (citywide overview to street-level)
- `-l flagged` — layer name in the tileset
- `--coalesce-densest-as-needed` — merge tiny polygons rather than dropping them
- `--force` — overwrite existing output

In [4]:
flagged_pmtiles = TILE_PATH / 'vacancy_flagged.pmtiles'

cmd = [
    'tippecanoe',
    '-o', str(flagged_pmtiles),
    '-l', 'flagged',
    '-n', 'PhillyStat360 Flagged Parcels (top 1%)',
    '-Z10', '-z16',
    '--coalesce-densest-as-needed',
    '--extend-zooms-if-still-dropping',
    '--force',
    str(FLAGGED_GEOJSON),
]
print(' '.join(cmd))
subprocess.run(cmd, check=True)
print(f'\nWrote {flagged_pmtiles} ({flagged_pmtiles.stat().st_size / 1e6:.1f} MB)')

tippecanoe -o /content/drive/MyDrive/PhillyStat_R/PhillyStat360/data_py/tiles/vacancy_flagged.pmtiles -l flagged -n PhillyStat360 Flagged Parcels (top 1%) -Z10 -z16 --coalesce-densest-as-needed --extend-zooms-if-still-dropping --force /content/drive/MyDrive/PhillyStat_R/PhillyStat360/data_py/vacancy_predictions_flagged.geojson

Wrote /content/drive/MyDrive/PhillyStat_R/PhillyStat360/data_py/tiles/vacancy_flagged.pmtiles (2.2 MB)


## 3. Tile the full prediction set

All 436K parcels. Polygons are small (single parcels) so we tile aggressively
at higher zooms and let tippecanoe drop / coalesce at low zooms.

- `-Z10 -z15` — zoom range
- `--drop-densest-as-needed` — at low zooms, drop the densest features
  (denser ZIPs lose some parcels, but coverage stays representative)
- `--extend-zooms-if-still-dropping` — push to higher zooms automatically
  if features still need dropping at the configured max
- `--no-tile-compression` is **off** (default = gzip), which is what most
  PMTiles serving infrastructure expects

On Colab CPU this takes ~3–7 minutes for the full set.

In [5]:
full_pmtiles = TILE_PATH / 'vacancy_predictions.pmtiles'

cmd = [
    'tippecanoe',
    '-o', str(full_pmtiles),
    '-l', 'parcels',
    '-n', 'PhillyStat360 Vacancy Predictions',
    '-Z10', '-z15',
    '--drop-densest-as-needed',
    '--extend-zooms-if-still-dropping',
    '--simplification=10',
    '--force',
    str(FULL_GEOJSON),
]
print(' '.join(cmd))
subprocess.run(cmd, check=True)
print(f'\nWrote {full_pmtiles} ({full_pmtiles.stat().st_size / 1e6:.1f} MB)')

tippecanoe -o /content/drive/MyDrive/PhillyStat_R/PhillyStat360/data_py/tiles/vacancy_predictions.pmtiles -l parcels -n PhillyStat360 Vacancy Predictions -Z10 -z15 --drop-densest-as-needed --extend-zooms-if-still-dropping --simplification=10 --force /content/drive/MyDrive/PhillyStat_R/PhillyStat360/data_py/vacancy_predictions.geojson

Wrote /content/drive/MyDrive/PhillyStat_R/PhillyStat360/data_py/tiles/vacancy_predictions.pmtiles (46.4 MB)


## 4. Verify outputs

Use `tippecanoe-decode` (ships with tippecanoe) to confirm metadata / layer
names / zoom range came through correctly.

In [6]:
for pmt in [flagged_pmtiles, full_pmtiles]:
    print(f'=== {pmt.name} ===')
    !tippecanoe-decode -s EPSG:4326 -c {pmt} 2>/dev/null | head -1
    !ls -lh {pmt}
    print()

=== vacancy_flagged.pmtiles ===
{ "type": "Feature", "tippecanoe": { "layer": "flagged", "minzoom": 10, "maxzoom": 10 }, "properties": { "parcel_number": "404241502", "ovs": 1, "data_split": "train", "ensemble_prob": 1, "ensemble_prob_raw": 0.9835228504316326, "risk_score": 100, "qtile_tier": "Top 1% (highest risk)", "ensemble_flag": 1, "rf_prob": 1, "logit_prob": 0.4205128205128205, "xgb_prob": 1, "lgb_prob": 1, "rf_flag": 1, "logit_flag": 1, "xgb_flag": 1, "lgb_flag": 1, "zip_code": 19153, "census_tract": 55, "geographic_ward": 40, "address": "7523 DICKENS PL" }, "geometry": { "type": "Polygon", "coordinates": [ [ [ -75.246477, 39.910395 ], [ -75.246220, 39.910263 ], [ -75.246305, 39.910065 ], [ -75.246391, 39.910131 ], [ -75.246477, 39.910395 ] ] ] } }
-rw------- 1 root root 2.1M Apr 30 17:44 /content/drive/MyDrive/PhillyStat_R/PhillyStat360/data_py/tiles/vacancy_flagged.pmtiles

=== vacancy_predictions.pmtiles ===
{ "type": "Feature", "tippecanoe": { "layer": "parcels", "minzoom": 

---
**Done.** Tilesets are in `data_py/tiles/`:

- `vacancy_flagged.pmtiles` — layer `flagged`, the top-1% inspection list
- `vacancy_predictions.pmtiles` — layer `parcels`, all 436K predictions

Each PMTile carries the export properties from §6 of `05_output_analysis.ipynb`
(`risk_score`, `ensemble_prob`, `qtile_tier`, `zip_code`, `census_tract`,
`geographic_ward`, `address`, etc.), so dashboards can style and filter without
fetching the source GeoJSON.

To preview locally, drop the `.pmtiles` URL into <https://protomaps.github.io/PMTiles/>.

In [ ]:
!jupyter nbconvert --to html /content/drive/MyDrive/PhillyStat_R/PhillyStat360/code/python/06_tiling.ipynb